In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import timm
from transformers import AutoModelForImageClassification, LayoutLMv3Processor, LayoutLMv3ForSequenceClassification

# --- 1. CONFIGURATION ---
# UBAH VARIABEL INI DI SETIAP NOTEBOOK: 'convnext', 'dit', atau 'layoutlmv3'
MODEL_TO_TRAIN = 'dit' 

CONFIG = {
    'train_dir': '/kaggle/input/competitions/document-image-classification-sd234504/train/train',
    'test_dir': '/kaggle/input/competitions/document-image-classification-sd234504/test/test',
    'train_csv': '/kaggle/input/competitions/document-image-classification-sd234504/train.csv',
    'working_dir': '/kaggle/working/',
    'num_classes': 16,
    'n_splits': 5,
    'batch_size': 8 if MODEL_TO_TRAIN == 'layoutlmv3' else 16, # LayoutLM butuh memori lebih besar
    'epochs': 15,
    'patience': 3, # Early stopping patience
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu')
}

import shutil

# Sesuaikan dengan nama path dataset checkpoint Anda di Kaggle
checkpoint_dataset_dir = '/kaggle/input/datasets/diftaradityapratama/dit-checkpoints/dit_base' 
working_dir = CONFIG['working_dir']

if os.path.exists(checkpoint_dataset_dir):
    for file in os.listdir(checkpoint_dataset_dir):
        if file.endswith('.npy'):
            src = os.path.join(checkpoint_dataset_dir, file)
            dst = os.path.join(working_dir, file)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
                print(f"Memuat kembali checkpoint: {file}")

# --- 2. PREPROCESSING: EASYOCR ---
def run_ocr():
    ocr_file = os.path.join(CONFIG['working_dir'], 'ocr_results.json')
    if os.path.exists(ocr_file):
        with open(ocr_file, 'r') as f:
            return json.load(f)
    
    print("Menjalankan EasyOCR...")
    import easyocr
    reader = easyocr.Reader(['en'], gpu=True)
    ocr_results = {}
    
    # Kumpulkan semua path
    all_images = []
    for root, dirs, files in os.walk(CONFIG['train_dir']):
        for file in files:
            if file.endswith('.tif'): all_images.append(os.path.join(root, file))
    for file in os.listdir(CONFIG['test_dir']):
        if file.endswith('.tif'): all_images.append(os.path.join(CONFIG['test_dir'], file))
        
    for img_path in tqdm(all_images, desc="OCR"):
        img_id = os.path.basename(img_path)
        result = reader.readtext(img_path)
        words, boxes = [], []
        for bbox, text, prob in result:
            # Format EasyOCR: [[x1,y1], [x2,y1], [x2,y2], [x1,y2]]
            # LayoutLM butuh: [xmin, ymin, xmax, ymax]
            xs = [pt[0] for pt in bbox]
            ys = [pt[1] for pt in bbox]
            boxes.append([int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))])
            words.append(text)
        ocr_results[img_id] = {'words': words, 'boxes': boxes}
        
    with open(ocr_file, 'w') as f:
        json.dump(ocr_results, f)
    return ocr_results

# Normalisasi Box untuk LayoutLM (skala 0-1000)
def normalize_bbox(bbox, width, height):
    return [
        int(1000 * (bbox[0] / width)),
        int(1000 * (bbox[1] / height)),
        int(1000 * (bbox[2] / width)),
        int(1000 * (bbox[3] / height))
    ]

# --- 3. DATASET CLASS ---
class DocDataset(Dataset):
    def __init__(self, df, img_dir, model_type, ocr_data=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.model_type = model_type
        self.ocr_data = ocr_data
        self.is_test = is_test
        
        if self.model_type == 'layoutlmv3':
            self.processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
        else:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Ekstrak hanya nama filenya saja dari CSV
        img_name = os.path.basename(row['filename'])
        
        # Gabungkan ulang dengan path folder class
        img_path = os.path.join(self.img_dir, img_name) if self.is_test else os.path.join(self.img_dir, row['class_name'], img_name)
        
        # --- PENANGANAN GAMBAR RUSAK (CORRUPT) ---
        try:
            image = Image.open(img_path).convert('RGB')
            width, height = image.size
        except Exception as e:
            # Jika gambar rusak, buat gambar hitam kosong (dummy) berukuran 224x224
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            width, height = 224, 224
        # -----------------------------------------
        
        if self.model_type == 'layoutlmv3':
            # Pastikan menggunakan img_name asli (tanpa path) sebagai key untuk ocr_data
            ocr = self.ocr_data.get(img_name, {'words': [], 'boxes': []})
            words = ocr['words'][:512] # Limit sequence length
            boxes = [normalize_bbox(b, width, height) for b in ocr['boxes'][:512]]
            
            # Handle empty OCR
            if len(words) == 0:
                words, boxes = [" "], [[0, 0, 0, 0]]
                
            encoding = self.processor(
                image, words, boxes=boxes, max_length=512, 
                padding="max_length", truncation=True, return_tensors="pt"
            )
            # Remove batch dimension added by processor
            encoding = {k: v.squeeze(0) for k, v in encoding.items()}
            if not self.is_test:
                encoding['labels'] = torch.tensor(row['label'])
            return encoding
            
        else:
            image = self.transform(image)
            if self.is_test:
                return image
            return image, torch.tensor(row['label'])

# --- 4. MODEL INITIALIZATION ---
def get_model(model_type):
    if model_type == 'convnext':
        return timm.create_model('convnext_base', pretrained=True, num_classes=CONFIG['num_classes'])
    elif model_type == 'dit':
        # Menggunakan Hugging Face Transformers untuk DiT
        return AutoModelForImageClassification.from_pretrained(
            "microsoft/dit-base", 
            num_labels=CONFIG['num_classes'],
            ignore_mismatched_sizes=True
        )
    elif model_type == 'layoutlmv3':
        return LayoutLMv3ForSequenceClassification.from_pretrained("microsoft/layoutlmv3-base", num_labels=CONFIG['num_classes'])
# --- 5. TRAINING LOOP (With Early Stopping & Fold Saving) ---
def train_model(model_type, train_df, test_df, ocr_data):
    oof_file = f"{CONFIG['working_dir']}oof_{model_type}.npy"
    test_file = f"{CONFIG['working_dir']}test_{model_type}.npy"
    
    # if os.path.exists(oof_file) and os.path.exists(test_file):
    #     print(f"File untuk {model_type} sudah lengkap. Selesai.")
    #     return
        
    skf = StratifiedKFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=42)
    
    # Load jika ada checkpoint yang tersimpan (jika terputus di tengah fold)
    oof_preds = np.load(oof_file) if os.path.exists(oof_file) else np.zeros((len(train_df), CONFIG['num_classes']))
    test_preds = np.load(test_file) if os.path.exists(test_file) else np.zeros((len(test_df), CONFIG['num_classes']))
    
    test_dataset = DocDataset(test_df, CONFIG['test_dir'], model_type, ocr_data, is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
        fold_oof_file = f"{CONFIG['working_dir']}oof_{model_type}_fold{fold}.npy"
        if os.path.exists(fold_oof_file):
            print(f"Fold {fold+1} sudah selesai sebelumnya, melewati...")
            # TAMBAHKAN BARIS INI: Isi array OOF dengan data fold yang di-skip
            oof_preds[val_idx] = np.load(fold_oof_file)
            continue
            
        print(f"\n--- Melatih {model_type.upper()} | Fold {fold+1}/{CONFIG['n_splits']} ---")
        train_sub = train_df.iloc[train_idx].reset_index(drop=True)
        val_sub = train_df.iloc[val_idx].reset_index(drop=True)
        
        train_ds = DocDataset(train_sub, CONFIG['train_dir'], model_type, ocr_data)
        val_ds = DocDataset(val_sub, CONFIG['train_dir'], model_type, ocr_data)
        
        train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False)
        
        model = get_model(model_type).to(CONFIG['device'])
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
        criterion = nn.CrossEntropyLoss()
        
        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None
        
        for epoch in range(CONFIG['epochs']):
            model.train()
            train_loss = 0
            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
                optimizer.zero_grad()
                
                if model_type == 'layoutlmv3':
                    batch = {k: v.to(CONFIG['device']) for k, v in batch.items()}
                    outputs = model(**batch)
                    loss = outputs.loss
                elif model_type == 'dit':
                    # Format Hugging Face menerima pixel_values dan labels
                    images, labels = batch[0].to(CONFIG['device']), batch[1].to(CONFIG['device'])
                    outputs = model(pixel_values=images, labels=labels)
                    loss = outputs.loss
                else:
                    images, labels = batch[0].to(CONFIG['device']), batch[1].to(CONFIG['device'])
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                
            # Validation
            model.eval()
            val_loss = 0
            val_probs = []
            with torch.no_grad():
                for batch in val_loader:
                    if model_type == 'layoutlmv3':
                        batch = {k: v.to(CONFIG['device']) for k, v in batch.items()}
                        outputs = model(**batch)
                        loss = outputs.loss
                        logits = outputs.logits
                    elif model_type == 'dit':
                        images, labels = batch[0].to(CONFIG['device']), batch[1].to(CONFIG['device'])
                        outputs = model(pixel_values=images, labels=labels)
                        loss = outputs.loss
                        logits = outputs.logits
                    else:
                        images, labels = batch[0].to(CONFIG['device']), batch[1].to(CONFIG['device'])
                        outputs = model(images)
                        loss = criterion(outputs, labels)
                        logits = outputs
                        
                    val_loss += loss.item()
                    probs = torch.softmax(logits, dim=1).cpu().numpy()
                    val_probs.extend(probs)
                    
            val_loss /= len(val_loader)
            print(f"Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f}")
            
            # Early Stopping Logic
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = model.state_dict()
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= CONFIG['patience']:
                    print(f"Early stopping aktif di Epoch {epoch+1}")
                    break
                    
        # Load best model untuk prediksi OOF & Test
        model.load_state_dict(best_model_state)
        model.eval()
        
        # Simpan prediksi OOF untuk fold ini
        oof_preds[val_idx] = np.array(val_probs)
        np.save(fold_oof_file, np.array(val_probs)) # Tandai fold selesai
        
        # Prediksi Test
        fold_test_probs = []
        with torch.no_grad():
            for batch in test_loader:
                if model_type == 'layoutlmv3':
                    batch = {k: v.to(CONFIG['device']) for k, v in batch.items()}
                    outputs = model(**batch)
                    logits = outputs.logits
                elif model_type == 'dit':
                    images = batch.to(CONFIG['device'])
                    outputs = model(pixel_values=images)
                    logits = outputs.logits
                else:
                    images = batch.to(CONFIG['device'])
                    logits = model(images)
                    
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                fold_test_probs.extend(probs)
                
        test_preds += np.array(fold_test_probs) / CONFIG['n_splits']
        
        # Simpan progress global secara real-time
        np.save(oof_file, oof_preds)
        np.save(test_file, test_preds)

# --- 6. EKSEKUSI UTAMA ---
if __name__ == "__main__":
    train_df = pd.read_csv(CONFIG['train_csv'])
    test_files = [f for f in os.listdir(CONFIG['test_dir']) if f.endswith('.tif')]
    test_df = pd.DataFrame({'filename': test_files})
    
    ocr_data = run_ocr() if MODEL_TO_TRAIN == 'layoutlmv3' else None
    
    # Ini hanya akan melatih model yang dipilih di awal script
    train_model(MODEL_TO_TRAIN, train_df, test_df, ocr_data)